<a href="https://colab.research.google.com/github/ZuhaaAsif/Applied-Search-Intelligence-Google-Search-Ranking-Discoverability/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZuhaaAsif/Applied-Search-Intelligence-Google-Search-Ranking-Discoverability/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup
%pip -q install duckdb huggingface_hub

import os, getpass
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

metrics = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(CASE WHEN gsc_sum_position >= gsc_impressions THEN gsc_sum_position ELSE 0 END) AS sum_position,
           SUM(CASE WHEN gsc_sum_position >= gsc_impressions THEN gsc_impressions ELSE 0 END) AS impressions_with_position
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
""").df()
metrics['avg_position'] = metrics['sum_position'] / metrics['impressions_with_position']
metrics['ctr'] = metrics['clicks'] / metrics['impressions']
print(f"{len(metrics):,} content items with real March impressions")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

176,738 content items with real March impressions


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule: A page is worth reviewing for CTR if it's getting real, trustworthy search visibility, sits at a position where similar pages earn meaningfully more clicks, and has enough volume that the gap isn't just noise.

Reason code: ctr_below_tier_expectation. Action: review_title_meta.

Signal 1: CTR vs. position tier. Weighted CTR by position bucket, computed on the position-bug-corrected data: 1-3 = 0.004088, 4-10 = 0.003222, 11-20 = 0.003126, 21-50 = 0.001398, 51+ = 0.000419, monotonically decreasing across all five tiers with no reversals. Verdict: CONFIRMED.

Before trusting avg_position for this signal, I found a real data-quality issue: on 2.7% impression-rows, affecting 38.7% content items, gsc_sum_position stayed flat while gsc_impressions spiked 20–100x, mathematically impossible for a real position, since any genuine average position ≥1 requires gsc_sum_position ≥ gsc_impressions. I excluded these specific rows from the position calculation rather than silently averaging over invalid values. Direct proof this mattered: content_44f34c0a90047651 moved from an impossible avg_position = 0.67 (wrongly landing it in tier 1-3) to a plausible 8.08 (tier 4-10) after the fix; content_9c057b66c30a3abb moved even further, from 0.12 to 13.29.

Signal 2: volume, gating trust in a CTR number at all. Percent of pages with exactly zero CTR by impression volume: 1-10 impressions = 96.9% zero-CTR, dropping steadily to 5000+ impressions = 1.8% zero-CTR.
Verdict: CONFIRMED — justifies the impressions >= 100 gate in the score formula below.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Signal 1 — CTR vs. position (flag-linked: this is the signal behind the starter pipeline's low_ctr_visible_page / ctr_review_candidate reason codes):
signal_df = metrics[metrics['avg_position'] > 0].copy()
signal_df['position_bin'] = pd.cut(
    signal_df['avg_position'], bins=[0, 3, 10, 20, 50, 100000],
    labels=['1-3', '4-10', '11-20', '21-50', '51+']
)

bucket_ctr = signal_df.groupby('position_bin', observed=True).agg(
    n=('content_hash_id', 'count'),
    total_impressions=('impressions', 'sum'),
    total_clicks=('clicks', 'sum')
)
bucket_ctr['weighted_ctr'] = bucket_ctr['total_clicks'] / bucket_ctr['total_impressions']
print(bucket_ctr)

                  n  total_impressions  total_clicks  weighted_ctr
position_bin                                                      
1-3           12373         37741176.0      154284.0      0.004088
4-10          84807        150967385.0      486467.0      0.003222
11-20         30864         31697586.0       99087.0      0.003126
21-50         33662         57857537.0       80910.0      0.001398
51+           13457          2370197.0         992.0      0.000419


In [3]:
# Signal 2 — volume, behind the quick-win-style logic (justifies a minimum-impressions gate before trusting a CTR number at all):
metrics['volume_bin'] = pd.cut(
    metrics['impressions'], bins=[0, 10, 50, 100, 500, 5000, metrics['impressions'].max() + 1],
    labels=['1-10', '11-50', '51-100', '101-500', '501-5000', '5000+'], include_lowest=True
)
vol_table = metrics.groupby('volume_bin', observed=True).agg(
    n=('content_hash_id', 'count'),
    pct_zero_ctr=('ctr', lambda s: (s == 0).mean())
)
print(vol_table)

                n  pct_zero_ctr
volume_bin                     
1-10        34910      0.969178
11-50       26105      0.921011
51-100      14491      0.856946
101-500     39356      0.681802
501-5000    48586      0.216811
5000+       13290      0.017983


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = has_volume × underperforming × ctr_gap × impressions; a plain multiply of readable gates and severity, no fitted weights, matching the rule above exactly. has_volume requires ≥100 impressions (Signal 2); underperforming requires a positive gap between a page's CTR and its position tier's expected CTR (Signal 1); the product ranks pages by how large and how well-supported the opportunity is. Every scored row carries reason_code = ctr_below_tier_expectation and action = review_title_meta. Writes 175,163 ranked rows to work/outputs/baseline_action_score.csv on each run.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
tier_expected = bucket_ctr['weighted_ctr'].to_dict()
signal_df['tier_expected_ctr'] = signal_df['position_bin'].map(tier_expected).astype(float)
signal_df['ctr_gap'] = signal_df['tier_expected_ctr'] - signal_df['ctr']

has_volume = (signal_df['impressions'] >= 100).astype(int)
underperforming = (signal_df['ctr_gap'] > 0).astype(int)

signal_df['score'] = has_volume * underperforming * signal_df['ctr_gap'] * signal_df['impressions']
signal_df['reason_code'] = 'ctr_below_tier_expectation'
signal_df['action'] = 'review_title_meta'

queue = signal_df.sort_values('score', ascending=False)
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv")
queue[['content_hash_id', 'position_bin', 'avg_position', 'ctr', 'tier_expected_ctr', 'ctr_gap', 'impressions', 'score', 'reason_code', 'action']].head(20)

Wrote 175,163 rows to work/outputs/baseline_action_score.csv


,content_hash_id,position_bin,avg_position,ctr,tier_expected_ctr,ctr_gap,impressions,score,reason_code,action
11993,content_44f34c0a90047651,4-10,8.081314,0.000113,0.003222,0.003109,212404.0,660.436156,ctr_below_tier_expectation,review_title_meta
128018,content_8d7d99f109e19aa2,1-3,2.468557,0.001420,0.004088,0.002668,203497.0,542.885343,ctr_below_tier_expectation,review_title_meta
165046,content_8e1334d6356668e3,4-10,4.761236,0.000007,0.003222,0.003215,134984.0,433.963231,ctr_below_tier_expectation,review_title_meta
70197,content_34a70fea29d15f24,4-10,3.182897,0.000301,0.003222,0.002922,143019.0,417.854667,ctr_below_tier_expectation,review_title_meta
76532,content_fec55986a1868d62,4-10,5.039321,0.000008,0.003222,0.003214,124075.0,398.810814,ctr_below_tier_expectation,review_title_meta
62651,content_7c6373141eae744a,4-10,5.948459,0.000626,0.003222,0.002596,132593.0,344.258636,ctr_below_tier_expectation,review_title_meta
106503,content_f6116743b00afc2d,4-10,9.735658,0.000139,0.003222,0.003083,107584.0,331.671340,ctr_below_tier_expectation,review_title_meta
39787,content_306bc78dff1eb683,1-3,1.604687,0.000433,0.004088,0.003655,80821.0,295.392120,ctr_below_tier_expectation,review_title_meta
62681,content_acbcc847f8996314,4-10,3.396293,0.001534,0.003222,0.001688,170808.0,288.400044,ctr_below_tier_expectation,review_title_meta
90467,content_cd3d932d4e1c8db0,4-10,7.831807,0.000045,0.003222,0.003178,89332.0,283.857341,ctr_below_tier_expectation,review_title_meta


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. ...44f34c (4-10, pos 8.08): CTR 0.0001 vs tier avg 0.0032, 212K impressions.
Wrong if: a featured snippet or "People Also Ask" box is siphoning clicks regardless of metadata — check the SERP before assuming a fix helps.
2. ...8d7d99 (1-3, pos 2.47): CTR 0.0014 vs 0.0041, 203K impressions. Wrong if: the gap is smaller than it looks, could be normal variance within the tier, not a real problem.
3.  ...8e1334 (4-10, pos 4.76): CTR near-zero (0.000007), 135K impressions.
Wrong if: this is a tracking/redirect issue rather than a content problem, unusual enough to check directly.
4. ...34a70f (4-10, pos 3.18): CTR 0.0003 vs 0.0032, 143K impressions.
Wrong if: it's a navigational/branded query where users already know the destination and don't need to click through.
5. ...fec55d (4-10, pos 5.04): CTR near-zero (0.000008), 124K impressions.
Wrong if: same near-zero-CTR signature as #3, worth checking as a shared pattern, not a one-off content issue.
6. ...7c6373 (4-10, pos 5.95): CTR 0.0006 vs 0.0032, 133K impressions.
Wrong if: nothing obvious, reasonable gap and volume, one of the more defensible flags as-is.
7. ...f61167 (4-10, pos 9.74): CTR 0.0001, 108K impressions.
Wrong if: position 9.74 sits right at the tier boundary, daily noise could push it into the next tier down, where this CTR would look expected.
8. ...306bc7 (1-3, pos 1.60): CTR 0.0004 vs 0.0041, 81K impressions. Wrong if: hard to see one, true top-3 with a large gap, one of the strongest candidates in the list.
9. ...acbcc8 (4-10, pos 3.40): CTR 0.0015 vs 0.0032, 171K impressions.
Wrong if: the gap is treated as severe, it's the smallest relative gap in the top 10 (CTR roughly half tier average), ranking high mostly on volume.
10. ...cd3d93 (4-10, pos 7.83): CTR near-zero (0.000045), 89K impressions. Wrong if: it's the third near-zero-CTR page seen so far, pattern is recurring enough to investigate directly rather than treat individually.
11. ...9ef3d7 (1-3, pos 2.41): CTR 0.0010 vs 0.0041, 89K impressions. Wrong if: nothing major, genuine top-3 gap, smaller volume than #8 but still reasonable.
12. ...b99ea6 (4-10, pos 4.55): CTR 0.0019 vs 0.0032, 194K impressions.
Wrong if: treated as a strong opportunity; smallest gap ratio in the top 20, ranks high on volume alone; a weak pick.
13. ...046fc4 (4-10, pos 7.21): CTR near-zero (0.000072), 84K impressions.
Wrong if: it's the fourth near-zero-CTR occurrence, worth a dedicated check for a shared broken tracking pixel or redirect.
14. ...9c057b (11-20, pos 13.29): CTR near-zero, 84K impressions, correctly re-tiered post position-fix.
Wrong if: the earlier data bug still has residual effects, this was one of the two rows most affected, worth double-checking specifically.
15. ...f43118 (4-10, pos 5.34): CTR 0.0014 vs 0.0032, 139K impressions.
Wrong if: nothing stands out; a moderate, defensible gap.
16. ...9540d8 (4-10, pos 8.01): CTR near-zero (0.000134), 82K impressions.
Wrong if: right at the tier boundary like #7; same boundary-sensitivity caveat applies.
17. ...c46df0 (1-3, pos 1.74): CTR 0.0006 vs 0.0041, 70K impressions. Wrong if: nothing major; genuine top-3 gap, lower volume than #8/#11 but still a real flag.
18. ...fc6767 (1-3, pos 2.13): CTR 0.0003 vs 0.0041, 60K impressions. Wrong if: the small sample matters, smallest volume of the top-3 group, so this is the lowest-confidence pick among them.
19. ...425715 (4-10, pos 6.98): CTR near-zero (0.000042), 72K impressions.
Wrong if: it's the fifth near-zero-CTR occurrence, at this point the pattern itself is the most useful thing to investigate, more than this single page.
20. ...36fc1e (4-10, pos 6.13): CTR 0.0002 vs 0.0032, 73K impressions.
Wrong if: nothing stands out; a moderate gap, unremarkable volume, an ordinary defensible flag.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Leakage check: no product-decision flags used anywhere (health_score, priority_score, action_type never touched); no future window, this rule scores the March 2026 snapshot only, nothing from a later period feeds in.

Weak picks: #12 (...b99ea6) ranks high mainly from raw volume (194K impressions) despite the smallest relative CTR gap in the top 20 — a case where the score formula's × impressions term overrides a genuinely modest opportunity. #9 (...acbcc8) shows the same pattern to a lesser degree.

A pattern worth flagging as a group, not five separate fixes: five of the top-20 pages (#3, #5, #10, #13, #19) share a near-identical near-zero CTR (0.00001–0.0001) despite six-figure impression volume. Treating these as five independent "rewrite the title" recommendations would be wrong, the shared signature suggests a common cause: a tracking issue, a SERP feature capturing clicks across similar query types, or a genuine content problem common to this page group. Flagged for grouped investigation rather than individually resolved.

Data-quality finding, carried from Section 1: the position-data bug was caught and corrected before it could distort this ranking, without the fix, two of the top-20 pages here would have been wrongly ranked in tier 1-3 instead of their real tiers (4-10 and 11-20).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pct_rows = 264737 / 9841378 * 100
pct_items = 68425 / 176738 * 100
print(f"{pct_rows:.2f}% of impression-rows affected")
print(f"{pct_items:.2f}% of content items have at least one affected day")

2.69% of impression-rows affected
38.72% of content items have at least one affected day


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.